In [2]:
from systemflow.hep.utils import hep_graph_from_spreadsheet, hep_with_updated_parameters, hep_construct_graph, dataframes_from_spreadsheet
from systemflow.hep.metrics import Productivity
from systemflow.classifier import get_passed
from systemflow.metrics import precision, recall, f1_score
from systemflow.models import density_scale_model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from multiprocess import Pool, cpu_count

In [4]:
# load data from the spreadsheet which defines the structure of the workflow,
# as well as the parameters for data rates, efficiency, data reduction, and classifier performance
run3_det, run3_proc, run3_globals = dataframes_from_spreadsheet("configurations/cms_system_60.xlsx")
run5_det, run5_proc, run5_globals = dataframes_from_spreadsheet("configurations/cms_system_200.xlsx")

In [5]:
run3_det

,Category,Detector,Data (bytes),Sample Rate,Link Efficiency (J/bit),Op Efficiency (J/op),Compression
0,Tracking,Inner Tracker,436666.666667,40000000,2.220000e-11,0,0
1,Tracking,Outer Tracker PS,206666.666667,40000000,2.220000e-11,0,0
2,Tracking,Outer Tracker 2S,126666.666667,40000000,2.220000e-11,0,0
3,Tracking,Track Finder TPG,10000.000000,40000000,2.220000e-11,0,0
4,Timing,MIP Timing BTL,76666.666667,40000000,2.220000e-11,0,0
5,Timing,MIP Timing ETL,136666.666667,40000000,2.220000e-11,0,0
6,Calorimetry,ECAL Barrel,180000.000000,40000000,2.220000e-11,0,0
7,Calorimetry,HCAL Barrel,240000.000000,40000000,2.220000e-11,0,0
8,Calorimetry,HCAL HO,30000.000000,40000000,2.220000e-11,0,0
9,Calorimetry,HCAL HF,60000.000000,40000000,2.220000e-11,0,0


In [6]:
run5_det

,Category,Detector,Data (bytes),Sample Rate,Compression,Link Efficiency (J/bit),Op Efficiency (J/op),PU 200
0,Tracking,Inner Tracker,1440000,40000000,0,2.220000e-11,0,1.440
1,Tracking,Outer Tracker PS,720000,40000000,0,2.220000e-11,0,0.720
2,Tracking,Outer Tracker 2S,430000,40000000,0,2.220000e-11,0,0.430
3,Tracking,Track Finder TPG,10000,40000000,0,2.220000e-11,0,0.010
4,Timing,MIP Timing BTL,240000,40000000,0,2.220000e-11,0,0.240
5,Timing,MIP Timing ETL,440000,40000000,0,2.220000e-11,0,0.440
6,Calorimetry,ECAL Barrel,600000,40000000,0,2.220000e-11,0,0.600
7,Calorimetry,HCAL Barrel,240000,40000000,0,2.220000e-11,0,0.240
8,Calorimetry,HCAL HO,30000,40000000,0,2.220000e-11,0,0.030
9,Calorimetry,HCAL HF,60000,40000000,0,2.220000e-11,0,0.060


In [7]:
run5_det.iloc[9]

Category                   Calorimetry
Detector                       HCAL HF
Data (bytes)                     60000
Sample Rate                   40000000
Compression                          0
Link Efficiency (J/bit)            0.0
Op Efficiency (J/op)                 0
PU 200                            0.06
Name: 9, dtype: object

In [9]:
#import the data predicting wall time scaling by pileup
scaling = pd.read_excel("wall time scaling.xlsx", sheet_name="Data")
#fit a polynomial to this data for CPU and GPU runtimes
fit_poly = lambda x, k3, k2, k1: k3 * x ** 3 + k2 * x ** 2 + k1 * x
k, cv = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time"])

In [10]:
#define a dictionary with functions defining the scaling of trigger runtimes with incoming data
funcs = {"Global": lambda x: fit_poly(x, *k), "Intermediate": lambda x: x / 2.0e6}

In [11]:
"""
Build a base system graph from the Run-3 spreadsheet data with interpolated
pileup detector data and a given L1T reduction ratio.
"""
def init_system(functions, l1t_reduction: float, pileup_interp: float):
    d = run3_det.copy()
    new_vals = (1 - pileup_interp) * d["Data (bytes)"].values + pileup_interp * run5_det["Data (bytes)"].values
    d["Data (bytes)"] = new_vals

    t = run3_proc.copy()
    # intermediate reduction stage
    t.at[4, "Reduction Ratio"] = l1t_reduction

    graph_def = hep_construct_graph(d, t, run3_globals, functions=functions)
    return graph_def

In [12]:
ex_baseline = init_system(funcs, 400, 0.0)

FileNotFoundError: [Errno 2] No such file or directory: '/home/wilkie/code/system_flow/HEP/HEP/l1t_data/egamma.csv'

In [13]:
# Execute the graph and inspect total power
ex_baseline_result = ex_baseline()
ex_baseline_result.metric_values["total power (W)"] / 1e6

NameError: name 'ex_baseline' is not defined

In [ ]:
ex_pu200 = init_system(funcs, 400, 1.0)

In [ ]:
ex_pu200_result = ex_pu200()
ex_pu200_result.metric_values["total power (W)"] / 1e6

In [ ]:
def extract_results(result):
    mv = result.metric_values
    power = mv["total power (W)"]
    confusion = mv["pipeline contingency (2x2)"]
    return power, confusion

In [ ]:
extract_results(ex_baseline_result)

In [ ]:
extract_results(ex_pu200_result)

In [ ]:
def vary_system(base_graph_def, reduction_ratio: float, interp: float):
    """Vary the system by updating detector data (pileup) and L1T reduction ratio.
    
    Uses hep_with_updated_parameters to efficiently update the base graph
    without recreating classifiers.
    """
    updates = {}
    for i in range(len(run3_det)):
        det = run3_det.iloc[i]
        name = det["Detector"]
        data = (1 - interp) * det["Data (bytes)"] + interp * run5_det.iloc[i]["Data (bytes)"]
        updates[name] = {"sample data (B)": data}
    updates["Intermediate"] = {"reduction ratio (1)": reduction_ratio}

    variant = hep_with_updated_parameters(base_graph_def, updates)
    result = variant()
    mv = result.metric_values
    power = mv["total power (W)"]
    confusion = mv["pipeline contingency (2x2)"]
    return power, confusion

In [ ]:
baseline = vary_system(ex_baseline, 400, 0.0)

In [ ]:
baseline

In [ ]:
run5 = vary_system(ex_baseline, 53.3, 1.0)

In [ ]:
run5

In [ ]:
#vary this accept rate from today's rate to the planned Run-5
l1t_reductions = np.linspace(450, 40, 101)
pileup = np.linspace(0.01, 1.0, 101)

In [ ]:
# Build base graph once (pileup=0, reduction=400)
# This creates classifiers once; hep_with_updated_parameters reuses them
base_def = init_system(funcs, 400, 0.0)

pmap_args = []
for s in pileup:
    for r in l1t_reductions:
        pmap_args.append((s, r))

In [ ]:
def map_fn(args):
    interp, reduction = args
    # Update detector data for this pileup level and reduction ratio
    updates = {}
    for i in range(len(run3_det)):
        det = run3_det.iloc[i]
        name = det["Detector"]
        data = (1 - interp) * det["Data (bytes)"] + interp * run5_det.iloc[i]["Data (bytes)"]
        updates[name] = {"sample data (B)": data}
    updates["Intermediate"] = {"reduction ratio (1)": reduction}

    variant = hep_with_updated_parameters(base_def, updates)
    result = variant()
    mv = result.metric_values
    return mv["total power (W)"], mv["pipeline contingency (2x2)"]

In [ ]:
n_cpus = cpu_count()

with Pool(n_cpus) as p:
    res = p.map(map_fn, pmap_args)

In [ ]:
res2 = [res[i:i+len(pileup)] for i in range(0, len(pileup)*len(l1t_reductions), len(pileup))]

In [ ]:
def sys_productivity(confusion, power):
    n = np.sum(get_passed(confusion))
    f1 = f1_score(confusion)
    productivity = (n * f1) / power
    return productivity

In [ ]:
def extract_metrics(results):
    all_confusion = np.array([r[1] for r in results])

    all_power = [r[0] / density_scale_model(2032) for r in results]
    all_power = np.array(all_power)

    all_recall = np.array([recall(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_precision = np.array([precision(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_f1 = np.array([f1_score(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])

    all_productivity = [sys_productivity(all_confusion[i,:,:], all_power[i]) for i in range(all_confusion.shape[0])]

    metrics = {"confusion": all_confusion,
               "power": all_power,
               "recall": all_recall,
               "precision": all_precision,
               "f1 score": all_f1,
               "productivity" : all_productivity}

    return metrics

In [ ]:
run5_metrics = [extract_metrics(r) for r in res2]

In [ ]:
res_f1 = np.stack([r["f1 score"] for r in run5_metrics]).transpose()

In [ ]:
res_recall = np.stack([r["recall"] for r in run5_metrics]).transpose()

In [ ]:
res_precision = np.stack([r["precision"] for r in run5_metrics]).transpose()

In [ ]:
power = np.stack([r["power"] for r in run5_metrics])

In [ ]:
power[1,1]

In [ ]:
res_productivity = np.stack([r["productivity"] for r in run5_metrics])

In [ ]:
from scipy.ndimage import gaussian_filter

In [ ]:
smoothed_f1 = gaussian_filter(res_f1, sigma=4)

In [ ]:
#np.savez_compressed("smoothed_f1.npz", smoothed_f1)

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=smoothed_f1,
        x=l1t_reductions, # horizontal axis
        y=pileup, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "F1 Score")
         
    ),
    )

y_offset = 0.015
fig.add_trace(go.Scatter(x = (400,),
                        y = (0.0 + y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol="circle"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(x = (53.3,),
                        y = (1.0 - y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "star"),
                        name = "Phase-2"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "Pileup",
                  title = "F1 Score by Pileup & Reduction Ratio",
                  legend=dict(xanchor = "right",
                    x = 0.95))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0.0, 0.8])
fig.show()

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=power,
        x=l1t_reductions, # horizontal axis
        y=pileup, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "Power (W)")
         
    ),
    )

y_offset = 0.015
fig.add_trace(go.Scatter(x = (400,),
                        y = (0.0 + y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol="circle"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(x = (53.3,),
                        y = (1.0 - y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "star"),
                        name = "Phase-2"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "Pileup",
                  title = "DAQ Power by Pileup & Reduction Ratio",
                  legend=dict(xanchor = "right",
                    x = 0.20,
                    y = 0.95))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0.0, 0.8])
fig.show()

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=res_productivity * 1000,
        x=l1t_reductions, # horizontal axis
        y=(pileup* 140)+60, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "Productivity (1/kJ)")
         
    ),
    )

y_offset = 3
fig.add_trace(go.Scatter(x = (400,),
                        y = (60 + y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "green", symbol="circle"),
                        name = "Run-3"))

fig.add_trace(go.Scatter(x = (53.3,),
                        y = (200 - y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "green", symbol = "star"),
                        name = "Run-5"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "Pileup",
                  title = "System Productivity by Pileup & L1T Reduction Ratio",
                  legend=dict(xanchor = "right",
                   x = 0.20,
                y = 0.95),)
fig.update_xaxes(autorange="reversed")
fig.add_annotation(x = -0.1, 
                   y = -0.1, 
                   showarrow=False,
                   text = "Baseline System (2032)", 
                   xref="paper", 
                   yref="paper",
                   font = dict(size = 14))
fig.show()

In [ ]:
fig.write_image(os.path.join("figures", "pileup rejection contours.png"))

In [ ]:
fig = go.Figure(data =
    go.Contour(
        z=smoothed_f1,
        x=l1t_reductions, # horizontal axis
        y=pileup, # vertical axis,
        contours = dict(showlabels = True),
        colorbar = dict(title = "F1 Score")
         
    ),
    )

y_offset = 0.015
fig.add_trace(go.Scatter(x = (400,),
                        y = (0.0 + y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol="circle"),
                        name = "Phase-1"))

fig.add_trace(go.Scatter(x = (53.3,),
                        y = (1.0 - y_offset,),
                        mode = "markers",
                        marker = dict(size = 14, color = "gray", symbol = "star"),
                        name = "Phase-2"))

fig.update_layout(width = 800, 
                  height = 600,
                  xaxis_title = "L1T Reduction Ratio",
                  yaxis_title = "Pileup",
                  title = "F1 Score by Pileup & Reduction Ratio",
                  legend=dict(xanchor = "right",
                    x = 0.95))
fig.update_xaxes(autorange="reversed")
fig.update_yaxes(range=[0.0, 0.8])
fig.show()

In [ ]:
#because its rejection is so much higher, there's more potential improvement gained by making L1T's skill higher
#than simply passing more data to the HLT